# Random Forest

In [ ]:
# Importar las librerías necesarias
from fastai.tabular.all import *
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score
import matplotlib.pyplot as plt
import seaborn as sns

## 1. Cargar y Preparar los Datos

Cargamos los datos previamente procesados del archivo pickle generado en el notebook de Prework.

In [ ]:
# Cargar datos preprocesados desde TabularPandas
to = load_pickle('./df_train-tabular-object.pkl')

# Obtener conjuntos de entrenamiento y validación
X_train = to.train.xs.copy()
y_train = to.train.y.copy()
X_test = to.valid.xs.copy()
y_test = to.valid.y.copy()

print(f"Tamaño conjunto de entrenamiento: {len(X_train)}")
print(f"Tamaño conjunto de validación: {len(X_test)}")
print(f"Número de características: {X_train.shape[1]}")

# Mostrar las primeras características
print("\nPrimeras características:")
print(X_train.columns.tolist()[:10])

# PATH = "/Users/luanagiusto/TP-1-ML"  # Cambia esto si tu path es diferente
# PATH = "C:/Users/julia/ML_TP"
PATH = "C:/Users/60083400/Desktop/DITELLA/ML"

## 2. Entrenar el Modelo Random Forest

Configuramos y entrenamos el modelo Random Forest con hiperparámetros básicos.

In [ ]:
# Crear y entrenar el modelo Random Forest
rf_model = RandomForestRegressor(
    n_estimators=100,
    max_depth=None,
    min_samples_split=2,
    min_samples_leaf=1,
    max_features='auto',
    random_state=42,
    n_jobs=-1  # Usar todos los cores disponibles
)

# Entrenar el modelo
rf_model.fit(X_train, y_train)

# Realizar predicciones
y_pred_train = rf_model.predict(X_train)
y_pred_test = rf_model.predict(X_test)

## 3. Evaluar el Modelo

Calculamos y visualizamos diferentes métricas de rendimiento.

In [ ]:
# Calcular métricas de rendimiento
train_mse = mean_squared_error(y_train, y_pred_train)
test_mse = mean_squared_error(y_test, y_pred_test)
train_rmse = np.sqrt(train_mse)
test_rmse = np.sqrt(test_mse)
train_r2 = r2_score(y_train, y_pred_train)
test_r2 = r2_score(y_test, y_pred_test)

print("Métricas de rendimiento:")
print(f"Train RMSE: {train_rmse:.2f}")
print(f"Test RMSE: {test_rmse:.2f}")
print(f"Train R²: {train_r2:.4f}")
print(f"Test R²: {test_r2:.4f}")

# Visualizar predicciones vs valores reales
plt.figure(figsize=(12, 5))

# Subplot para conjunto de entrenamiento
plt.subplot(1, 2, 1)
plt.scatter(y_train, y_pred_train, alpha=0.5)
plt.plot([y_train.min(), y_train.max()], [y_train.min(), y_train.max()], 'r--', lw=2)
plt.xlabel('Valores Reales')
plt.ylabel('Predicciones')
plt.title('Predicciones vs Valores Reales (Train)')

# Subplot para conjunto de prueba
plt.subplot(1, 2, 2)
plt.scatter(y_test, y_pred_test, alpha=0.5)
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2)
plt.xlabel('Valores Reales')
plt.ylabel('Predicciones')
plt.title('Predicciones vs Valores Reales (Test)')

plt.tight_layout()
plt.show()

## 4. Análisis de Importancia de Características

Visualizamos las características más importantes según el modelo.

In [ ]:
# Calcular importancia de características
importances = pd.Series(
    rf_model.feature_importances_,
    index=X_train.columns
).sort_values(ascending=True)

# Visualizar las 15 características más importantes
plt.figure(figsize=(10, 8))
importances.tail(15).plot(kind='barh')
plt.title('Top 15 Características más Importantes')
plt.xlabel('Importancia Relativa')
plt.tight_layout()
plt.show()

# Imprimir las 10 características más importantes
print("\nTop 10 características más importantes:")
for name, importance in importances.tail(10)[::-1].items():
    print(f"{name}: {importance:.4f}")

## 5. Model Selection

In [ ]:
# Nueva celda markdown

# Nueva celda código
from sklearn.model_selection import KFold
import random

def random_cv_rf(X, y, param_distributions, n_iter=15, sample_size=0.3, n_splits=3, random_state=42):
    """
    Búsqueda aleatoria de hiperparámetros con validación cruzada y muestra reducida
    """
    # Tomar muestra aleatoria del dataset
    np.random.seed(random_state)
    sample_size = int(len(X) * sample_size)
    indices = np.random.choice(len(X), size=sample_size, replace=False)
    X_sample = X.iloc[indices]
    y_sample = y.iloc[indices]
    
    print(f"Tamaño original del dataset: {len(X)}")
    print(f"Tamaño de la muestra: {len(X_sample)}")
    
    # Inicializar K-Fold
    kf = KFold(n_splits=n_splits, shuffle=True, random_state=random_state)
    
    # Lista para almacenar resultados
    results = []
    
    # Generar combinaciones aleatorias de parámetros
    for i in range(n_iter):
        params = {
            key: random.choice(value) if isinstance(value, list) else 
                 int(np.random.uniform(value[0], value[1])) if isinstance(value, tuple) else value
            for key, value in param_distributions.items()
        }
        
        fold_scores = []
        
        # K-Fold Cross Validation
        for fold, (train_idx, val_idx) in enumerate(kf.split(X_sample), 1):
            X_train_fold = X_sample.iloc[train_idx]
            X_val_fold = X_sample.iloc[val_idx]
            y_train_fold = y_sample.iloc[train_idx]
            y_val_fold = y_sample.iloc[val_idx]
            
            # Entrenar modelo
            model = RandomForestRegressor(
                **params,
                random_state=random_state,
                n_jobs=-1
            )
            
            model.fit(X_train_fold, y_train_fold)
            
            # Predecir y calcular RMSE
            y_pred = model.predict(X_val_fold)
            rmse = np.sqrt(mean_squared_error(y_val_fold, y_pred))
            fold_scores.append(rmse)
        
        # Calcular media y desviación estándar del RMSE
        mean_rmse = np.mean(fold_scores)
        std_rmse = np.std(fold_scores)
        
        results.append({
            'params': params,
            'mean_rmse': mean_rmse,
            'std_rmse': std_rmse,
            'fold_scores': fold_scores
        })
        
        print(f"\nIteración {i+1}/{n_iter}")
        print(f"Parámetros: {params}")
        print(f"RMSE medio: {mean_rmse:.2f} (±{std_rmse:.2f})")
        print("Scores por fold:", fold_scores)
        print("-" * 80)
    
    # Ordenar resultados por RMSE medio
    results.sort(key=lambda x: x['mean_rmse'])
    return results

# Nueva celda código
# Definir distribuciones de hiperparámetros
param_distributions = {
    'n_estimators': [100, 200, 300],
    'max_depth': [None, 10, 20, 30],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
    'max_features': ['auto', 'sqrt', 'log2']
}

# Ejecutar búsqueda aleatoria con validación cruzada
results = random_cv_rf(
    X_train, y_train,
    param_distributions,
    n_iter=15,
    sample_size=0.3,
    n_splits=3,
    random_state=42
)

# Mostrar los mejores resultados
print("\nMejores 3 combinaciones de hiperparámetros:")
for i, result in enumerate(results[:3], 1):
    print(f"\n{i}. Mejores parámetros:")
    print(f"   {result['params']}")
    print(f"   RMSE medio: {result['mean_rmse']:.2f} (±{result['std_rmse']:.2f})")
    print(f"   Scores por fold: {result['fold_scores']}")

# Nueva celda código
# Entrenar modelo final con los mejores hiperparámetros
best_params = results[0]['params']
rf_model = RandomForestRegressor(**best_params, random_state=42, n_jobs=-1)
rf_model.fit(X_train, y_train)

# Realizar predicciones y calcular métricas
y_pred_train = rf_model.predict(X_train)
y_pred_test = rf_model.predict(X_test)

## 6. Análisis de Errores

Analizamos la distribución de los errores y buscamos patrones.

In [ ]:
# Calcular errores
train_errors = y_train - y_pred_train
test_errors = y_test - y_pred_test

plt.figure(figsize=(12, 5))

# Histograma de errores
plt.subplot(1, 2, 1)
plt.hist(test_errors, bins=50, alpha=0.7, label='Test')
plt.hist(train_errors, bins=50, alpha=0.7, label='Train')
plt.xlabel('Error de Predicción')
plt.ylabel('Frecuencia')
plt.title('Distribución de Errores')
plt.legend()

# Scatter plot de errores vs valores reales
plt.subplot(1, 2, 2)
plt.scatter(y_test, test_errors, alpha=0.5)
plt.axhline(y=0, color='r', linestyle='--')
plt.xlabel('Valores Reales')
plt.ylabel('Error de Predicción')
plt.title('Errores vs Valores Reales (Test)')

plt.tight_layout()
plt.show()

# Estadísticas de los errores
print("\nEstadísticas de los errores (Test):")
print(pd.Series(test_errors).describe())

## 6. Guardar el Modelo

Guardamos el modelo entrenado para su uso posterior.

In [ ]:
import pickle

# Guardar el modelo
with open('random_forest_model.pkl', 'wb') as file:
    pickle.dump(rf_model, file)

print("Modelo guardado como 'random_forest_model.pkl'")

#### Predict & submission

In [ ]:
# 1. Preparar df_test
df_test = pd.read_parquet(os.path.join(PATH, "prework_test_output.parquet"))

# mantener dtypes como hiciste
for col in df_test.select_dtypes("float64"):
    df_test[col] = df_test[col].astype("float32")

# guardar ids
ids = df_test["sk_id_curr"].copy()

# 2. Eliminar columna target si existe
if "target" in df_test.columns:
    df_test = df_test.drop(columns=["target"])

# 3. Alinear columnas con las del train (X_train fue tu DataFrame de features de entrenamiento)
train_cols = list(X_train.columns)

# Añadir columnas faltantes con 0
for c in train_cols:
    if c not in df_test.columns:
        df_test[c] = 0.0

# Eliminar columnas extra en test (excepto el id)
extra_cols = [c for c in df_test.columns if c not in train_cols and c != "sk_id_curr"]
if extra_cols:
    df_test = df_test.drop(columns=extra_cols)

# Reordenar columnas para que tengan exactamente el mismo orden que en train
df_test = df_test[train_cols]

# 4. Comprobar NaNs y tipos que podrían romper predict
assert df_test.isna().sum().sum() == 0, "Hay NaNs en df_test; imputar antes de predecir"
for col in df_test.select_dtypes("float64"):
    df_test[col] = df_test[col].astype("float32")

# 5. Predecir con el modelo optimizado
y_pred_test = rf_model.predict(df_test)

# 6. Crear archivo de salida
out = pd.DataFrame({"SK_ID_CURR": ids.values, "TARGET": y_pred_test})
out.to_csv(os.path.join(PATH, "predictions_test_rf_optimized.csv"), index=False)
print("Predicciones guardadas en:", os.path.join(PATH, "predictions_test_rf_optimized.csv"))